In [42]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path
import pandera as pa
from pandera import Column, DataFrameSchema, Check

from common.loaders import load_valid_country_ids

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,capital,capitalInfo"

# --- Esquema de validación ---
capitals_schema = DataFrameSchema({
    "cca3": Column(
        str,
        checks=[
            Check.str_length(3, 3),                     # debe tener 3 letras
            Check.isin(load_valid_country_ids())        # debe existir en countries.csv
        ],
        nullable=False
    ),
    "capital": Column(str, nullable=False),
    "lat": Column(
        float,
        checks=[Check.ge(-90), Check.le(90)],
        nullable=True
    ),  # puede venir vacío
    "lng": Column(
        float,
        checks=[Check.ge(-180), Check.le(180)],
        nullable=True
    ),  # puede venir vacío
})


@task
def extract_capitals():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()

@task
def transform_capitals(data):
    df = pd.DataFrame(data)

    # Extraer lat/lng
    df["lat"] = df["capitalInfo"].apply(
        lambda x: x["latlng"][0] if x and "latlng" in x and len(x["latlng"]) > 0 else None
    )
    df["lng"] = df["capitalInfo"].apply(
        lambda x: x["latlng"][1] if x and "latlng" in x and len(x["latlng"]) > 1 else None
    )

    # Explode capitals en filas separadas
    df_exp = df.explode("capital").reset_index(drop=True)

    # Limpiar
    df_exp = df_exp.dropna(subset=["capital"])
    df_exp = df_exp[["cca3", "capital", "lat", "lng"]]


    return df_exp

@task
def validate_capitals(df: pd.DataFrame) -> pd.DataFrame:
    return capitals_schema.validate(df)

@task
def load_capitals(df: pd.DataFrame):
    file_path = Path("../stage/country_capitals.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f" Saved {len(df)} rows to {file_path}")

@flow(name="etl_capitals_flow")
def etl_capitals():
    data = extract_capitals()
    df = transform_capitals(data)
    df = validate_capitals(df)  # validación con pandera
    load_capitals(df)

if __name__ == "__main__":
    etl_capitals()


03:33:40.286 | INFO    | Flow run 'ginger-wombat' - Beginning flow run 'ginger-wombat' for flow 'etl_capitals_flow'

03:33:41.437 | INFO    | Task run 'extract_capitals-2c7' - Finished in state Completed()

03:33:41.676 | INFO    | Task run 'transform_capitals-0e6' - Finished in state Completed()

03:33:41.905 | INFO    | Task run 'validate_capitals-32f' - Finished in state Completed()

 Saved 249 rows to ..\stage\country_capitals.csv


03:33:42.127 | INFO    | Task run 'load_capitals-6f4' - Finished in state Completed()

03:33:42.138 | INFO    | Flow run 'ginger-wombat' - Finished in state Completed()